In [1]:
%pip install -q "pandas>=2.3,<3.0" anndata scanpy
import numpy as np
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_parquet('/content/drive/MyDrive/Datasets/fydp/tcga_brca_tri_modal_dataset.parquet')  # adjust folder path if inside a subfolder
df.head()

Mounted at /content/drive


,vital_status,event,survival_time,age_at_index,ajcc_pathologic_stage,ER_status,PR_status,HER2_status,days_to_death,days_to_last_follow_up,...,ENSG00000288661.1,ENSG00000288662.1,ENSG00000288663.1,ENSG00000288665.1,ENSG00000288667.1,ENSG00000288669.1,ENSG00000288670.1,ENSG00000288671.1,ENSG00000288674.1,ENSG00000288675.1
patient_id,,,,,,,,,,,,,,,,,,,,,
TCGA-3C-AAAU,Alive,0.0,4047.0,55.0,Stage X,Positive,Positive,Negative,NaN,4047.0,...,0.0,0.0000,0.5171,0.0,0.0000,0.0000,30.172701,0.0,0.1323,1.4353
TCGA-3C-AALI,Alive,0.0,4005.0,50.0,Stage IIB,Positive,Positive,Positive,NaN,4005.0,...,0.0,1.5146,0.7494,0.0,0.0000,0.0228,13.930200,0.0,0.0850,2.4341
TCGA-3C-AALJ,Alive,0.0,1474.0,62.0,Stage IIB,Positive,Positive,Equivocal,NaN,1474.0,...,0.0,2.3066,0.8500,0.0,0.0000,0.0000,11.335200,0.0,0.0719,2.2242
TCGA-3C-AALK,Alive,0.0,1448.0,52.0,Stage IA,Positive,Positive,Positive,NaN,1448.0,...,0.0,0.0000,0.5027,0.0,1.0673,0.0000,9.980100,0.0,0.0947,2.0927
TCGA-4H-AAAK,Alive,0.0,348.0,50.0,Stage IIIA,Positive,Positive,Equivocal,NaN,348.0,...,0.0,0.0000,0.5134,0.0,0.0000,0.0000,11.657800,0.0,0.0521,1.7913


In [3]:
df.shape

(1073, 60771)

In [4]:
df.isna().sum()

,0
vital_status,0
event,0
survival_time,0
age_at_index,0
ajcc_pathologic_stage,11
...,...
ENSG00000288669.1,0
ENSG00000288670.1,0
ENSG00000288671.1,0
ENSG00000288674.1,0


In [5]:
cut_idx = df.columns.get_loc('MUT_ANKRD12') + 1
df_rna = df.iloc[:, cut_idx:]
df_rna.shape

(1073, 60660)

In [6]:
print('Total missing values:', df_rna.isna().sum().sum())
print('Columns with missing values:', (df_rna.isna().sum() > 0).sum())
print('Duplicate rows:', df_rna.duplicated().sum())
print('Duplicate patient IDs:', df_rna.index.duplicated().sum())

Total missing values: 0
Columns with missing values: 0
Duplicate rows: 0
Duplicate patient IDs: 0


In [7]:
from huggingface_hub import login
from transformers import AutoModel, AutoTokenizer

login()

model_name = 'ScientaLab/eva-rna'
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(model_name, trust_remote_code=True, device_map='auto')
model.eval()
print('Model loaded on:', next(model.parameters()).device)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

configuration_eva_rna.py:   0%|          | 0.00/1.04k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/ScientaLab/eva-rna:
- configuration_eva_rna.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/183 [00:00<?, ?B/s]

tokenization_eva_rna.py:   0%|          | 0.00/9.10k [00:00<?, ?B/s]

... (6 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  194MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/319 [00:00<?, ?it/s]

Model loaded on: cuda:0


In [12]:
import anndata as ad
import scanpy as sc

# 1. Strip Ensembl version suffix
base_genes = df_rna.columns.str.split(".").str[0]
df_rna_unver = df_rna.copy()
df_rna_unver.columns = base_genes
if df_rna_unver.columns.duplicated().any():
    df_rna_unver = df_rna_unver.T.groupby(level=0).sum().T

# 2. Keep only genes in eva-rna tokenizer vocabulary
ensembl_mapper = tokenizer.gene_mapper["ensembl_to_ncbi"]
mapped_genes = [g for g in df_rna_unver.columns if g in ensembl_mapper]
print(f"Total unversioned genes: {df_rna_unver.shape[1]} | Matched in eva-rna: {len(mapped_genes)}")
df_rna_eva = df_rna_unver[mapped_genes]

# 3. Build AnnData
adata = ad.AnnData(
    X=df_rna_eva.to_numpy(dtype="float32"),
    obs=pd.DataFrame(index=df_rna_eva.index),
    var=pd.DataFrame(index=df_rna_eva.columns),
)

# 4. Normalise to 1e6 (data is already TPM but scanpy HVG needs raw-ish counts)
#    Log1p first so HVG selection uses seurat_v3 (expects raw counts — use flavor="cell_ranger" instead)
sc.pp.log1p(adata, base=2)

# 5. Select top 2000 highly variable genes (reduces seq length from 26525 -> 2000)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="cell_ranger")
adata = adata[:, adata.var.highly_variable].copy()
print(f"AnnData after HVG selection: {adata.shape}")


Total unversioned genes: 60616 | Matched in eva-rna: 26525
AnnData after HVG selection: (1073, 2000)


In [ ]:
import torch, os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

with torch.no_grad():
    embeddings = model.encode_anndata(
        tokenizer, adata,
        species="human",
        batch_size=16,
    )

embeddings = embeddings.astype("float32")
print("Embeddings shape:", embeddings.shape)

emb_cols = [f"eva_{i}" for i in range(embeddings.shape[1])]
df_eva_emb = pd.DataFrame(embeddings, index=adata.obs_names, columns=emb_cols)
df_eva_emb.to_parquet("rna_eva_embeddings.parquet")
df_eva_emb


/root/.cache/huggingface/modules/transformers_modules/ScientaLab/eva_hyphen_rna/4232edf3c7efb01e4bb828be016f3fc86600f3e6/modeling_eva_rna.py:330: UserWarning: Auto-converted 2000/2000 genes from ensembl to NCBI GeneIDs (species: human). Set species='human'/'mouse' explicitly if auto-detection is wrong.
  return encode_from_anndata(
Encoding samples: 100%|██████████| 68/68 [05:18<00:00,  4.68s/batch]

Embeddings shape: (1073, 256)


,eva_0,eva_1,eva_2,eva_3,eva_4,eva_5,eva_6,eva_7,eva_8,eva_9,...,eva_246,eva_247,eva_248,eva_249,eva_250,eva_251,eva_252,eva_253,eva_254,eva_255
patient_id,,,,,,,,,,,,,,,,,,,,,
TCGA-3C-AAAU,-2.153962,-0.290481,0.180694,-3.529985,-0.276185,-2.094843,1.503977,1.295508,0.813222,1.153410,...,0.676140,-2.982211,0.253861,-1.616247,-0.910579,-0.235289,1.863513,-0.556452,0.512754,-2.518935
TCGA-3C-AALI,-2.385773,-0.096298,0.089126,-3.397844,-0.400117,-2.101587,1.651369,0.475417,1.201100,1.197020,...,0.016930,-2.994860,-0.274166,-1.670812,-0.850337,-0.230574,1.801920,0.157671,0.613829,-2.156768
TCGA-3C-AALJ,-2.599488,0.209992,-0.119050,-3.401907,-0.407471,-2.551439,1.605150,0.648154,1.347954,1.274194,...,0.272354,-3.190524,-0.181975,-1.639181,-0.475116,0.001570,2.054497,-0.049078,0.624193,-2.281215
TCGA-3C-AALK,-2.704945,0.192047,-0.477967,-3.628271,-0.768288,-2.002047,1.785265,0.034251,1.394154,1.500029,...,0.321800,-3.123309,-0.354690,-1.647089,-0.367320,-0.038733,2.225318,-0.240957,0.519407,-2.078643
TCGA-4H-AAAK,-2.698491,0.295609,-0.278496,-3.456106,-0.677330,-2.160398,1.714190,-0.029962,1.111928,1.634661,...,0.544121,-3.377841,-0.196960,-1.723608,-0.148143,0.056772,2.072745,-0.724219,0.637411,-2.129378


In [16]:
df_eva_emb

,eva_0,eva_1,eva_2,eva_3,eva_4,eva_5,eva_6,eva_7,eva_8,eva_9,...,eva_246,eva_247,eva_248,eva_249,eva_250,eva_251,eva_252,eva_253,eva_254,eva_255
patient_id,,,,,,,,,,,,,,,,,,,,,
TCGA-3C-AAAU,-2.153962,-0.290481,0.180694,-3.529985,-0.276185,-2.094843,1.503977,1.295508,0.813222,1.153410,...,0.676140,-2.982211,0.253861,-1.616247,-0.910579,-0.235289,1.863513,-0.556452,0.512754,-2.518935
TCGA-3C-AALI,-2.385773,-0.096298,0.089126,-3.397844,-0.400117,-2.101587,1.651369,0.475417,1.201100,1.197020,...,0.016930,-2.994860,-0.274166,-1.670812,-0.850337,-0.230574,1.801920,0.157671,0.613829,-2.156768
TCGA-3C-AALJ,-2.599488,0.209992,-0.119050,-3.401907,-0.407471,-2.551439,1.605150,0.648154,1.347954,1.274194,...,0.272354,-3.190524,-0.181975,-1.639181,-0.475116,0.001570,2.054497,-0.049078,0.624193,-2.281215
TCGA-3C-AALK,-2.704945,0.192047,-0.477967,-3.628271,-0.768288,-2.002047,1.785265,0.034251,1.394154,1.500029,...,0.321800,-3.123309,-0.354690,-1.647089,-0.367320,-0.038733,2.225318,-0.240957,0.519407,-2.078643
TCGA-4H-AAAK,-2.698491,0.295609,-0.278496,-3.456106,-0.677330,-2.160398,1.714190,-0.029962,1.111928,1.634661,...,0.544121,-3.377841,-0.196960,-1.723608,-0.148143,0.056772,2.072745,-0.724219,0.637411,-2.129378
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-WT-AB44,-3.454087,1.097275,0.095867,-3.189383,-0.645928,-2.028559,1.554853,0.421946,1.232571,1.405390,...,0.825279,-3.028241,-0.338061,-1.741979,-0.586726,-0.158987,1.259997,-0.573157,-0.345522,-1.472014
TCGA-XX-A899,-2.578705,0.284645,-0.581849,-3.698812,-0.853371,-2.301447,1.793718,0.470274,1.413289,0.946639,...,0.430162,-3.293477,-0.092788,-1.608129,-0.413531,0.244333,2.094165,-0.234404,0.457212,-2.472640
TCGA-XX-A89A,-3.205063,0.069611,-0.343750,-3.438967,-0.879204,-2.488165,1.838579,0.198752,1.779164,1.113951,...,0.650124,-2.634417,-0.024688,-2.024450,-0.268428,-0.384085,1.924380,-0.061316,0.922178,-2.080115
